<a href="https://colab.research.google.com/github/asipnana/NaturalLanguageProcessing/blob/main/Model3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# MODEL 3: IndoBERT Fine-Tuning (Weighted CrossEntropy)
# Optimized for Imbalanced Data & Google Colab Stability
# Integrasi KaggleHub untuk Download Dataset Otomatis
# ============================================================

# 1. INSTALL
!pip install transformers datasets accelerate kagglehub -q

# 2. IMPORT
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import re
import string
import os
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding
)
from datasets import Dataset
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

# 3. CEK GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 4. LOAD DATA DARI KAGGLE
print("📥 Downloading dataset from Kaggle...")
path = kagglehub.dataset_download("salmanabdu/tokopedia-product-reviews-2025")
file_path = os.path.join(path, 'tokopedia_product_reviews_2025.csv')

df = pd.read_csv(file_path)
df = df[['review_text', 'sentiment_label']].copy()
df.dropna(inplace=True)

# Filter minimal kata
df['word_count'] = df['review_text'].astype(str).apply(lambda x: len(x.split()))
df = df[df['word_count'] >= 3]

# 5. REFINED PREPROCESSING
def clean_text_refined(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'&\w+;', '', text)
    # ANGKA DIPERTAHANKAN karena penting untuk konteks rating
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_review'] = df['review_text'].apply(clean_text_refined)
df = df[df['clean_review'].str.len() > 0]

# 6. ENCODE & SPLIT
le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment_label'])
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'], df['label'], test_size=0.2, stratify=df['label'], random_state=42
)

# 7. DAMPENED CLASS WEIGHTS (LOGIC FIX)
class_counts = Counter(y_train)
counts = [class_counts[i] for i in range(len(le.classes_))]
total = sum(counts)

# Sqrt Inverse Frequency agar bobot tidak terlalu ekstrim
weights = [np.sqrt(total / c) for c in counts]
weights_norm = [w / sum(weights) * len(counts) for w in weights]
class_weights = torch.tensor(weights_norm, dtype=torch.float32).to(device)

print(f"Mapping: {dict(zip(le.classes_, range(len(le.classes_))))}")
print(f"Class Weights: {weights_norm}")

# 8. TOKENIZER & DATASET
MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(examples['text'], padding=False, truncation=True, max_length=128)

train_ds = Dataset.from_pandas(pd.DataFrame({'text': X_train.values, 'label': y_train.values}))
test_ds = Dataset.from_pandas(pd.DataFrame({'text': X_test.values, 'label': y_test.values}))
train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

# 9. CUSTOM TRAINER
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# 10. MODEL
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

# 11. TRAINING ARGS
training_args = TrainingArguments(
    output_dir='./indobert_weighted_results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=200,
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    report_to="none",
    logging_steps=50,
    save_total_limit=1
)

# 12. TRAIN
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=lambda p: {
        'accuracy': accuracy_score(p.label_ids, p.predictions.argmax(-1)),
        'f1_macro': f1_score(p.label_ids, p.predictions.argmax(-1), average='macro')
    },
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("\n🚀 Training IndoBERT Model 3 (Weighted CE)...")
trainer.train()

# 13. EVALUATE
print("\n📊 Final Evaluation:")
predictions = trainer.predict(test_ds)
y_pred = predictions.predictions.argmax(-1)
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# 14. SAVE
trainer.save_model('./indobert_model3_weighted')
tokenizer.save_pretrained('./indobert_model3_weighted')
print("\n✅ Saved to ./indobert_model3_weighted")

Using device: cpu
📥 Downloading dataset from Kaggle...


100%|██████████| 3.54M/3.54M [00:00<00:00, 30.5MB/s]

Extracting files...


Mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
Class Weights: [np.float64(1.4089749584055462), np.float64(1.427606841374079), np.float64(0.16341820022037487)]


config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🚀 Training IndoBERT Model 3 (Weighted CE)...


Step,Training Loss,Validation Loss


KeyboardInterrupt: 